# Module 2 Demo: Photometric Stereo + Poisson Integration

本 Notebook 使用**合成数据**演示从法向量到 3D 高度场的完整重建流程，无需任何外部数据即可运行。

In [ ]:
import sys
sys.path.insert(0, '../02_Visuo_Tactile_Perception')

import numpy as np
import matplotlib.pyplot as plt
from photometric_stereo import photometric_stereo
from fft_poisson_integration import reconstruct_height_from_normals, normals_to_gradients
from multigrid_poisson_integration import multigrid_poisson_integration, compare_methods

## 1. 生成合成高度场

In [ ]:
H, W = 128, 128
x = np.linspace(-2, 2, W)
y = np.linspace(-2, 2, H)
X, Y = np.meshgrid(x, y)

# 真实高度场：双高斯峰
z_true = 2.0 * np.exp(-(X**2 + Y**2) / 1.5) - 0.8 * np.exp(-((X-1)**2 + (Y+0.5)**2) / 0.8)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
im = ax.imshow(z_true, cmap='jet', extent=[-2,2,-2,2])
ax.set_title('Ground Truth Height Map')
plt.colorbar(im, ax=ax, label='Height [a.u.]')
plt.tight_layout()
plt.show()

## 2. 从高度场生成法向量（模拟光度立体视觉输入）

In [ ]:
# 数值梯度
dz_dx = np.gradient(z_true, axis=1)
dz_dy = np.gradient(z_true, axis=0)

# 法向量：n = (-dz/dx, -dz/dy, 1) / ||...||
nx = -dz_dx
ny = -dz_dy
nz = np.ones_like(z_true)
norm = np.sqrt(nx**2 + ny**2 + nz**2)
normals = np.stack([nx/norm, ny/norm, nz/norm], axis=-1)

# 可视化法向量 RGB 编码
normal_vis = ((normals + 1.0) / 2.0 * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(normal_vis)
axes[0].set_title('Surface Normals (RGB)')
axes[1].quiver(X[::8,::8], Y[::8,::8], normals[::8,::8,0], normals[::8,::8,1],
               scale=20, color='b')
axes[1].set_title('Normal Vector Field')
axes[1].set_aspect('equal')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 3. FFT 泊松积分重建

In [ ]:
import time

p, q = normals_to_gradients(normals)

t0 = time.perf_counter()
z_fft = reconstruct_height_from_normals(normals, pixel_size_mm=1.0)
t_fft = time.perf_counter() - t0

print(f'FFT reconstruction: {t_fft*1000:.2f} ms')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(z_true, cmap='jet')
axes[0].set_title('Ground Truth')
axes[1].imshow(z_fft, cmap='jet')
axes[1].set_title(f'FFT Reconstruction ({t_fft*1000:.1f}ms)')
error = z_fft - z_true
error -= np.mean(error)  # 去均值
axes[2].imshow(error, cmap='RdBu_r', vmin=-0.1, vmax=0.1)
axes[2].set_title(f'Reconstruction Error (RMSE={np.sqrt(np.mean(error**2)):.4f})')
plt.tight_layout()
plt.show()

## 4. FFT vs Multigrid 对比

In [ ]:
results = compare_methods(p, q)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(z_true, cmap='jet')
axes[0].set_title('Ground Truth')
axes[1].imshow(results['z_fft'], cmap='jet')
axes[1].set_title(f"FFT ({results['t_fft_ms']:.1f}ms)")
axes[2].imshow(results['z_mg'], cmap='jet')
axes[2].set_title(f"Multigrid ({results['t_mg_ms']:.1f}ms)")
plt.tight_layout()
plt.show()